# Data Transformation
**Manual Feature Engineering: Daily Return**

Author: Pete King

This notebook computes daily returns for all sector ETFs.  Daily return represents a change in the price of an asset and is given by the formula:

**return = log(price_t / price_t-1)**

We will later construct a PyTorch DataSet that will give us the flexibility to use daily return data to generate samples and labels for training our deep learning models, using flexible time windows.  Part of what we want to investigate is whether an LSTM can learn patterns in both near-term and long-term return data to predict future return and volatility for an asset.

In [12]:
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_prep as dp

ETF_DATA_FILE = 'etf_raw_data.csv'
INDEX_DATA_FILE = 'filled_indicies_data.csv'

## I. Import data and compute daily return

In [16]:
etf_df = pd.read_csv(
    ETF_DATA_FILE,
    index_col='date',
    parse_dates=True
)
tickers = etf_df.columns

In [17]:
# Find starting date for each asset for future reference
starting_date = {}
for ticker in tickers:
    starting_date[ticker] = etf_df[ticker].dropna().index.min()
sorted([(k, v) for k, v in starting_date.items()], key=lambda i:i[1])

[('SPY', Timestamp('1993-01-29 00:00:00')),
 ('XLB', Timestamp('1998-12-22 00:00:00')),
 ('XLE', Timestamp('1998-12-22 00:00:00')),
 ('XLF', Timestamp('1998-12-22 00:00:00')),
 ('XLI', Timestamp('1998-12-22 00:00:00')),
 ('XLK', Timestamp('1998-12-22 00:00:00')),
 ('XLP', Timestamp('1998-12-22 00:00:00')),
 ('XLU', Timestamp('1998-12-22 00:00:00')),
 ('XLV', Timestamp('1998-12-22 00:00:00')),
 ('XLY', Timestamp('1998-12-22 00:00:00')),
 ('QQQ', Timestamp('1999-03-10 00:00:00')),
 ('IWM', Timestamp('2000-05-26 00:00:00')),
 ('IEF', Timestamp('2002-07-30 00:00:00')),
 ('LQD', Timestamp('2002-07-30 00:00:00')),
 ('TLT', Timestamp('2002-07-30 00:00:00')),
 ('TIP', Timestamp('2003-12-05 00:00:00')),
 ('GLD', Timestamp('2004-11-18 00:00:00')),
 ('BND', Timestamp('2007-04-10 00:00:00')),
 ('HYG', Timestamp('2007-04-11 00:00:00')),
 ('BIL', Timestamp('2007-05-30 00:00:00')),
 ('XLRE', Timestamp('2015-10-08 00:00:00'))]

In [18]:
for ticker in tickers:
    etf_df[ticker + '_return'] = (
        np.log(etf_df[ticker] / etf_df[ticker].shift(1))
    )
etf_df

,BIL,BND,GLD,HYG,IEF,IWM,LQD,QQQ,SPY,TIP,...,XLB_return,XLE_return,XLF_return,XLI_return,XLK_return,XLP_return,XLRE_return,XLU_return,XLV_return,XLY_return
date,,,,,,,,,,,,,,,,,,,,,
1993-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.241400,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.413815,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.465536,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.724159,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1993-02-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24.827616,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-13,91.510002,73.550003,460.839996,79.199997,95.589996,246.152130,108.169998,593.719971,662.289978,110.709999,...,-0.009912,0.003298,0.001228,-0.003577,-0.007574,0.005799,0.002607,0.009844,-0.002467,-0.005936
2026-03-16,91.510002,73.830002,460.429993,79.449997,96.019997,248.477997,108.690002,600.380005,669.030029,111.059998,...,0.004260,0.003460,0.008351,0.008527,0.014370,0.002828,0.007780,0.006368,0.008112,0.012015
2026-03-17,91.519997,73.980003,459.269989,79.809998,96.190002,250.050003,109.300003,603.309998,670.789978,111.449997,...,0.002426,0.010480,0.005260,0.002646,0.005461,-0.003300,0.003283,-0.002754,-0.009114,0.008697


In [19]:
etf_df.to_csv('etf_with_return.csv', index_label='date')